In [29]:
import ast
import pandas as pd
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import math
from src.metrics import *

In [30]:
raw_data = []

data_set = "YAP1"
infile = f"../Data/Protein_Gym_Datasets/{data_set}.csv"

data = []
with open(infile, "r") as f:
    for line in f.readlines()[1:]:
        line = line.split(",")[0:7]
        mutant_name = line[0]
        sequence = line[1]
        normalized_score = round(float(line[3]), 3)
        ddG_score = round(float(line[4]), 3)
        data.append([mutant_name, sequence, normalized_score, ddG_score])

In [31]:
"""Distribution of of number of mutations"""
mutation_counts = dict()
for mutant in data:
    number_of_mutations = 1
    for symbol in mutant[0]:
        if symbol == ":":
            number_of_mutations += 1
    if number_of_mutations not in mutation_counts.keys():
        mutation_counts.update({number_of_mutations: 1})
    else:
        mutation_counts.update({number_of_mutations: mutation_counts[number_of_mutations] + 1})

sum_all_mutation_counts = 0
for key in mutation_counts.keys():
    sum_all_mutation_counts += mutation_counts[key] * key
print(f"total number of mutations: {sum_all_mutation_counts}")

total number of mutations: 20174


In [32]:
mutation_counts = dict(sorted(mutation_counts.items(), key=lambda x: x[0], reverse=False))

mutation_counts_plt = make_subplots(rows=1, cols=1)
for key in mutation_counts.keys():
    mutation_counts_plt.append_trace(
    go.Bar(x=[key], y=[mutation_counts[key]], text=[mutation_counts[key]], textposition='outside'), row=1,
    col=1)
    mutation_counts_plt.update_layout(
    title_text=f"[{data_set}] Number of Mutations and their Counts",
    title_font=dict(color="black", size=20),
    showlegend=False,
    paper_bgcolor='rgb(233,233,233)',
    plot_bgcolor='rgb(233,233,233)',
    width=1000,
    )

mutation_counts_plt.update_traces(textangle=0, textposition="auto", textfont=dict(size=12, color="black"),
cliponaxis=False)
mutation_counts_plt.update_layout(uniformtext_minsize=10, uniformtext_mode='show')

mutation_counts_plt.update_yaxes(dict(
range=[0, 1.1*max(mutation_counts.values())],
color='black',
showgrid=False,
linecolor='black',
gridcolor='grey',
griddash="dot",
gridwidth=0.5)
)

mutation_counts_plt.update_xaxes(dict(color='black',
showgrid=True,
gridcolor='lightgrey',
griddash="dot",
showline=True,
linecolor='black',
zerolinecolor='black',
linewidth=2,
gridwidth=1,
dtick=1
),
) 

In [33]:
mutation_counts_plt.write_image(f"../Data/Protein_Gym_Datasets/{data_set}_Mutations_Counts.jpg")

In [34]:
"""Position_of_mutations"""
positions_count = dict()
for sample in data:
    number_of_mutations = 1
    mutant = sample[0].split(":")
    if mutant == 'WT' or mutant =='wAT' or mutant =='wt':
        continue

    for mutation in mutant:
        try:
            position = int(mutation[1:-1])
        except Exception:
            print(mutant)
            continue

        if position not in positions_count.keys():
            positions_count.update({position: 1})
        else:
            positions_count.update({position: positions_count[position] + 1})

sum_all_position_counts = sum(positions_count.values())
print(f"total number of mutations: {sum_all_position_counts}")


total number of mutations: 20174


In [35]:
positions_count = dict(sorted(positions_count.items(), key=lambda x: x[0], reverse=False))

positions_count_plt = make_subplots(rows=1, cols=1)
for key in positions_count.keys():
    positions_count_plt.append_trace(
        go.Bar(x=[key], y=[positions_count[key]], text=[positions_count[key]], textposition='auto'), row=1,
        col=1)
positions_count_plt.update_layout(
    title_text=f"[{data_set}] # Mutations for every AA-Sequence Position",
    title_font=dict(color="black", size=20),
    showlegend=False,
    paper_bgcolor='rgb(233,233,233)',
    plot_bgcolor='rgb(233,233,233)',
    width=2000,
    height=800
)
# positions_count_plt.update_traces(textangle=300, textposition="outside", textfont=dict(size=12, color="black"),
#                                   cliponaxis=False)
# positions_count_plt.update_layout(uniformtext_minsize=10, uniformtext_mode='show')
positions_count_plt.update_yaxes(dict(
    color='black',
    showgrid=True,
    minor_griddash="dot",
    gridcolor='grey',
    griddash="dot",
    linecolor='black',
    gridwidth=0.5,

)
)
positions_count_plt.update_xaxes(dict(color='black',
                                      minor=dict(ticklen=2, tickcolor='grey', showgrid=True, gridcolor='lightgrey',
                                                 dtick=1),
                                      dtick=5,
                                      tickangle=45,
                                      ticklen=10,
                                      tickson="boundaries",
                                      ticklabelmode="period",
                                      showgrid=True,
                                      gridcolor='lightgrey',
                                      griddash="dot",
                                      minor_griddash="dot",
                                      showline=True,
                                      zerolinecolor='black',
                                      linewidth=1,
                                      gridwidth=1,
                                      linecolor='black',
                                      ),
                                 )

In [36]:
positions_count_plt.write_image(f"../Data/Protein_Gym_Datasets/{data_set}_Position_Counts.jpg")

In [37]:
import numpy as np
import pandas as pd
mutants_df = pd.read_csv(infile)

# Bin the Norm_Score_1 values of the filtered dataframe
threshold = 1
score_bins = 40
bin_size = round((mutants_df["Norm_Score_1"].max() - mutants_df["Norm_Score_1"].min()) / score_bins,3)
# Define bin edges from 0 to 1 in steps of 0.05
bin_edges = np.arange(0, (1 + bin_size), bin_size)
# bin_labels = [f"({round(bin_edges[i], 3)}, {round(bin_edges[i+1], 3)}]" for i in range(len(bin_edges)-1)]

# filtered_df = mutants_df.nsmallest(int(threshold * len(mutants_df)), "ΔΔG").copy()
# filtered_df['score_bin'] = pd.cut(filtered_df['Norm_Score_1'], bins=bin_edges, include_lowest=False) #labels=bin_labels,

# Count the number of points in each score bin


mutants_df['score_bin'] = pd.cut(mutants_df['Norm_Score_1'], bins=bin_edges, include_lowest=False) #labels=bin_labels,
score_bin_counts = mutants_df["score_bin"].value_counts().sort_index()

# Plot as a bar chart
fig = go.Figure(
    data=go.Bar(
        x=score_bin_counts.index.astype(str),
        y=score_bin_counts.values,
        text=score_bin_counts.values,
        textposition='outside'
    )
)

fig.update_layout(
    xaxis_title=f"Norm_Score_1 bins (lowest {int(threshold*100)}% ΔΔG, bin size 0.05)",
    yaxis_title="Count",
    title=f"Distribution of Norm_Score_1 (lowest {int(threshold*100)}% ΔΔG, bin size 0.05)"
)
fig.show()

In [38]:
'''Creating a Heatmap of Normalized Score vs Number of Mutations, heat = amount of datapoints in each bin''' 
# Define the score ranges as provided by the user
min_score = min([score[2] for score in data])  # Get the minimum normalized score
max_score = max([score[2] for score in data])  # Get the maximum normalized score

#obtain_number of mutations for each mutant
number_mutations = []
for mutant in data:
    n_mutations = 1
    for symbol in mutant[0]:
        if symbol == ":":
            n_mutations += 1
    number_mutations.append(n_mutations)
x_vals = number_mutations
y_vals = [score[2] for score in data]

num_bins = 20
y_bins = np.linspace(min_score, max_score, num_bins + 1)

# Ensure y_bins are rounded for display purposes
y_bins = np.round(y_bins, 2)

y_bin_labels = [f"{y_bins[i]} - {y_bins[i+1]}" for i in range(len(y_bins) - 1)]

heatmap = go.Histogram2d(
    x=x_vals,
    y=y_vals,
    xbins=dict(start=min(x_vals), end=max(x_vals) + 1, size=1),
    ybins=dict(start=y_bins[0], end=y_bins[-1], size=(y_bins[1] - y_bins[0])),
    colorscale=[
        [0, 'rgba(0,0,0,0)'],
        [0.00001, 'rgba(0,0,255,0.8)'],
        [0.15, 'rgba(255,0,150,0.8)'],
        [0.5, 'rgba(255,0,0,0.8)'],
        [0.75, 'rgba(255,100,0,0.8)'],
        [1, 'rgba(255,255,0,0.8)'],
    ],
    colorbar=dict(title="# Datapoints"),
    opacity=0.9
)

histogram = make_subplots(rows=1, cols=1)
histogram.add_trace(heatmap, row=1, col=1)
histogram.update_layout(
    title="Number of Mutations vs Normalized Score (Heatmap)",
    xaxis_title="Number of Mutations",
    yaxis_title="Normalized Score",
    width=900,
    height=700,
    paper_bgcolor='rgb(233,233,233)',
    plot_bgcolor='rgb(233,233,233)',
)

# Calculate tick values for the center of each bin
tick_vals_center = [(y_bins[i] + y_bins[i+1]) / 2 for i in range(len(y_bins) - 1)]

histogram.update_yaxes(
    tickvals=tick_vals_center,
    ticktext=y_bin_labels,
    title="Normalized Score Bin Range",
    color='black',
    # Try to force all ticks by setting tickmode to 'array' and tick0 to the first bin center
    # Also, set dtick to the bin size to ensure consistent spacing
    tickmode='array',
    dtick=(y_bins[1] - y_bins[0]),
)

histogram.show()

In [39]:
histogram.write_image(f"../Data/Protein_Gym_Datasets/{data_set}_Mutations_vs_Score.jpg")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Extract DMS and ZS scores
zs_scores = [entry[3] for entry in data]
dms_scores = [entry[2] for entry in data]

# Define threshold
threshold = 0.5

# Group data based on threshold
good_points = [(zs, dms) for zs, dms in zip(zs_scores, dms_scores) if zs <= threshold]
bad_points = [(zs, dms) for zs, dms in zip(zs_scores, dms_scores) if zs > threshold]

# DMS scores for boxplots
good_dms = [dms for _, dms in good_points]
bad_dms = [dms for _, dms in bad_points]

# Compute means
mean_good = sum(good_dms) / len(good_dms) if good_dms else 0
mean_bad = sum(bad_dms) / len(bad_dms) if bad_dms else 0

# Create subplots
dms_vs_zs = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        f"DMS Score vs ZS Score (ρ = {pearson_correlation(dms_scores, zs_scores):.2f})",
        f"DMS Score by ZS Score Threshold ({threshold} kcal/mol)"
    )
)

# Scatterplot
dms_vs_zs.add_trace(go.Scatter(
    x=[zs for zs, _ in good_points],
    y=[dms for _, dms in good_points],
    mode='markers',
    marker=dict(color='green', opacity=0.4),
    name=f'ZS ≤ {threshold}',
), row=1, col=1)

dms_vs_zs.add_trace(go.Scatter(
    x=[zs for zs, _ in bad_points],
    y=[dms for _, dms in bad_points],
    mode='markers',
    marker=dict(color='red', opacity=0.4),
    name=f'ZS > {threshold}',
), row=1, col=1)

# Boxplots
dms_vs_zs.add_trace(go.Box(
    y=bad_dms,
    boxmean=True,
    name=f'ZS > {threshold} (n={len(bad_dms)})',
    marker_color='red',
), row=1, col=2)

dms_vs_zs.add_trace(go.Box(
    y=good_dms,
    boxmean=True,
    name=f'ZS ≤ {threshold} (n={len(good_dms)})',
    marker_color='green',
), row=1, col=2)

dms_vs_zs.update_layout(
    title=f"Comparison of Normalized DMS Score vs ZS Score (ΔΔG) for {data_set} Dataset",
    title_font=dict(size=22, color='black'),
    width=1400,
    height=750,
    showlegend=False,
    legend=dict(x=0.5, y=1.15, orientation='h', xanchor='center'),
    paper_bgcolor='rgb(233,233,233)',
    plot_bgcolor='rgb(233,233,233)',
)

# Axis styling
for col in [1, 2]:
    dms_vs_zs.update_xaxes(showline=True, linecolor='black', linewidth=1, row=1, col=col)
    dms_vs_zs.update_yaxes(showline=True, linecolor='black', linewidth=1, row=1, col=col)
    dms_vs_zs.update_yaxes(range=[0, None], row=1, col=col)

# Axis labels
dms_vs_zs.update_xaxes(title='ZS Score (ΔΔG)', row=1, col=1)
dms_vs_zs.update_yaxes(title='Normalized DMS Score', row=1, col=1)
dms_vs_zs.update_yaxes(title='Normalized DMS Score', row=1, col=2)

dms_vs_zs.show()

In [41]:
dms_vs_zs.write_image(f"../Data/Protein_Gym_Datasets/{data_set}_DMS_vs_ZS_th-{threshold}.jpg")